In [ ]:
!pip install xgboost

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


 1. Load raw training and testing data

In [ ]:
X_train_raw = pd.read_csv("X_train_raw.csv")
X_test_raw = pd.read_csv("X_test_raw.csv")

y_train = pd.read_csv("y_train.csv")
y_test = pd.read_csv("y_test.csv")

2. Create target variable: Age = Rings + 1.5

In [ ]:
y_train_age = y_train["Rings"] + 1.5
y_test_age = y_test["Rings"] + 1.5


3. Prepare raw X data for XGBoost


In [ ]:
X_train_xgb = X_train_raw.copy()
X_test_xgb = X_test_raw.copy()

bool_cols = X_train_xgb.select_dtypes(include=["bool"]).columns

X_train_xgb[bool_cols] = X_train_xgb[bool_cols].astype(int)
X_test_xgb[bool_cols] = X_test_xgb[bool_cols].astype(int)

4. Train XGBoost Regressor

In [ ]:
xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

xgb_model.fit(X_train_xgb, y_train_age)


5. Predict age on testing data

In [ ]:
xgb_pred_age = xgb_model.predict(X_test_xgb)

6. Evaluate model performance

In [ ]:
mae = mean_absolute_error(y_test_age, xgb_pred_age)
mse = mean_squared_error(y_test_age, xgb_pred_age)
r2 = r2_score(y_test_age, xgb_pred_age)

results = pd.DataFrame({
    "Model": ["XGBoost raw"],
    "MAE": [mae],
    "MSE": [mse],
    "R2": [r2]
})

print("\n================ XGBoost Model Performance ================")
print(results)

7. Save prediction results

In [ ]:
predictions = X_test_raw.copy()
predictions["Actual_Rings"] = y_test["Rings"]
predictions["Actual_Age"] = y_test_age
predictions["Predicted_Age_XGBoost"] = xgb_pred_age
predictions["Residual"] = predictions["Actual_Age"] - predictions["Predicted_Age_XGBoost"]
predictions["Absolute_Error"] = abs(predictions["Residual"])

predictions.to_csv("xgboost_predictions.csv", index=False)
results.to_csv("xgboost_model_performance.csv", index=False)

8. Feature importance

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X_train_xgb.columns,
    "Importance": xgb_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print("\n================ XGBoost Feature Importance ================")
print(feature_importance)

feature_importance.to_csv("xgboost_feature_importance.csv", index=False)
# Feature importance plot
plt.figure(figsize=(9, 6))
plt.barh(feature_importance["Feature"], feature_importance["Importance"])
plt.gca().invert_yaxis()
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("XGBoost Feature Importance")
plt.tight_layout()
plt.savefig("xgboost_feature_importance.png", dpi=300)
plt.show()

9. Age range error analysis

In [ ]:
predictions["Age_Range"] = pd.cut(
    predictions["Actual_Age"],
    bins=[0, 7.5, 10.5, 13.5, np.inf],
    labels=[
        "Young age <= 7.5",
        "Medium age 8.5-10.5",
        "Old age 11.5-13.5",
        "Very old age >= 14.5"
    ]
)

age_range_errors = predictions.groupby("Age_Range", observed=False).agg(
    Sample_Size=("Actual_Age", "count"),
    MAE=("Absolute_Error", "mean"),
    MSE=("Residual", lambda x: np.mean(x ** 2)),
    Mean_Actual_Age=("Actual_Age", "mean"),
    Mean_Predicted_Age=("Predicted_Age_XGBoost", "mean")
).reset_index()

print("\n================ XGBoost Error by Age Range ================")
print(age_range_errors)

age_range_errors.to_csv("xgboost_age_range_errors.csv", index=False)


# Age range error plot
plt.figure(figsize=(9, 5))
plt.bar(age_range_errors["Age_Range"].astype(str), age_range_errors["MAE"])
plt.xlabel("Age Range")
plt.ylabel("MAE")
plt.title("XGBoost Prediction Error by Age Range")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig("xgboost_age_range_errors.png", dpi=300)
plt.show()

10. Save the trained XGBoost model

In [ ]:
xgb_model.save_model("xgboost_abalone_age_model.json")

11. Short investigation summary for report writing

In [ ]:
top_features = feature_importance.head(5)["Feature"].tolist()
most_difficult_range = age_range_errors.sort_values("MAE", ascending=False).iloc[0]

summary_text = f"""
XGBoost Investigation Summary

1. Model used:
XGBoost Regressor was used to predict abalone age using raw physical measurements.

2. Target variable:
Age = Rings + 1.5

3. Model performance:
MAE = {mae:.3f}
MSE = {mse:.3f}
R2 = {r2:.3f}

4. Important physical measurements:
The top five most important features in the XGBoost model were:
{top_features}

5. Challenging age range:
The most difficult age range to predict was:
{most_difficult_range["Age_Range"]}
with MAE = {most_difficult_range["MAE"]:.3f}

"""

with open("xgboost_investigation_summary.txt", "w", encoding="utf-8") as f:
    f.write(summary_text)

print("\n================ Investigation Summary ================")
print(summary_text)

print("\nAll XGBoost output files have been saved successfully.")
